# v8 SFT Candidate Policy

This notebook downloads `candidates_v7.csv` from Hugging Face with `HF_TOKEN`, trains the supervised warm-start policy for v8, then uploads artifacts to `devaanshpa/orbit-wars-agent/v7/sft`.

In [ ]:
from getpass import getpass
import os

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required because this notebook uploads v8 SFT artifacts")
    os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd()
if not (repo_root / "notebooks" / "v8" / "train_sft_policy.py").exists() and (repo_root.parent.parent / "notebooks" / "v8" / "train_sft_policy.py").exists():
    repo_root = repo_root.parent.parent

data_path = os.environ.get("CANDIDATES_CSV", "").strip()
remote_data = os.environ.get("V8_SFT_DATA_REMOTE_PATH", "").strip()
if data_path:
    print("Using explicit local CANDIDATES_CSV override:", data_path)
elif remote_data:
    print("SFT will download candidate CSV from Hugging Face:", remote_data)
else:
    print("SFT will download the newest data/*/candidates_v7.csv from Hugging Face.")

print("repo_root:", repo_root)

In [ ]:
import importlib.util
import sys

missing = [pkg for pkg in ("torch", "huggingface_hub", "matplotlib") if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing packages:", missing)
    print("Install with:")
    print(f"{sys.executable} -m pip install torch huggingface-hub matplotlib")
    raise RuntimeError("Install missing packages, restart the kernel if needed, then rerun this notebook.")

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Defaults tuned for a 1000-game both-sides dataset on Kaggle 2*T4.
# Override any value by setting the matching environment variable before this cell.
sft_epochs = os.environ.get("V8_SFT_EPOCHS", "180")
sft_batch_groups = os.environ.get("V8_SFT_BATCH_GROUPS", "192")
sft_lr = os.environ.get("V8_SFT_LR", "0.00065")
sft_weight_decay = os.environ.get("V8_SFT_WEIGHT_DECAY", "0.00025")
sft_dropout = os.environ.get("V8_SFT_DROPOUT", "0.14")
sft_bce_weight = os.environ.get("V8_SFT_BCE_WEIGHT", "0.28")
sft_pair_weight = os.environ.get("V8_SFT_PAIR_WEIGHT", "0.25")
sft_rank_weight = os.environ.get("V8_SFT_RANK_WEIGHT", "0.42")
sft_ensemble_size = os.environ.get("V8_SFT_ENSEMBLE_SIZE", "3")
sft_patience = os.environ.get("V8_SFT_PATIENCE", "28")

print("SFT config:", {
    "epochs": sft_epochs,
    "batch_groups": sft_batch_groups,
    "lr": sft_lr,
    "weight_decay": sft_weight_decay,
    "dropout": sft_dropout,
    "bce_weight": sft_bce_weight,
    "pair_weight": sft_pair_weight,
    "rank_weight": sft_rank_weight,
    "ensemble_size": sft_ensemble_size,
    "patience": sft_patience,
})

cmd = [
    sys.executable,
    str(repo_root / "notebooks" / "v8" / "train_sft_policy.py"),
    "--export-dir",
    str(repo_root / "notebooks" / "v8" / "exports" / "sft"),
    "--epochs",
    sft_epochs,
    "--batch-groups",
    sft_batch_groups,
    "--lr",
    sft_lr,
    "--weight-decay",
    sft_weight_decay,
    "--dropout",
    sft_dropout,
    "--bce-weight",
    sft_bce_weight,
    "--pair-weight",
    sft_pair_weight,
    "--rank-weight",
    sft_rank_weight,
    "--ensemble-size",
    sft_ensemble_size,
    "--patience",
    sft_patience,
    "--upload",
]
if os.environ.get("CANDIDATES_CSV"):
    cmd.extend(["--csv", os.environ["CANDIDATES_CSV"]])
if os.environ.get("V8_SFT_DATA_REMOTE_PATH"):
    cmd.extend(["--data-remote-path", os.environ["V8_SFT_DATA_REMOTE_PATH"]])

print("Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=str(repo_root), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)